# CityFlo Bus Service — Metro Cities: Exploratory Data Analysis

**Dataset:** `cityflo_bus_service_metro_cities.csv` (3,258 rows × 36 columns)
**Workflow followed:** *Exploratory Data Analysis — A Standard 23-Step Checklist for Any Dataset*
(Classroom Computer Institute — Data Analytics)

This notebook walks through **all 23 steps** of the checklist, in order, across four phases:

| Phase | Steps | Goal |
|---|---|---|
| Phase 1 — Inspect | 1–8 | Understand the shape, structure, and quality of the raw data |
| Phase 2 — Clean & Prepare | 9–17 | Organize columns, clean values, export a final clean dataset |
| Phase 3 — Analyze | 18–21 | Explore relationships and statistically test them |
| Phase 4 — Report | 22–23 | Define KPIs and charts for the final dashboard |

Each step below has its own markdown explanation followed by the code that performs it.


In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
import numpy as np

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 50)

RAW_PATH = "/kaggle/input/datasets/rishijmanna/cityflo-bus-service-metro-cities-cleaned/cityflo_bus_service_metro_cities_cleaned.csv"
df = pd.read_csv(RAW_PATH)
df.head()

,trip_id,booking_id,customer_id,customer_name,gender,age,city,route_id,route_name,origin_stop,destination_stop,bus_number,bus_type,driver_id,driver_name,trip_date,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,distance_km,fare_inr,discount_inr,payment_mode,booking_channel,seat_number,trip_status,cancellation_reason,rating,occupancy_pct,weather,is_peak_hour,subscription_type,device_type,gps_enabled,complaint_raised
0,TRP000768,BK47984879,CUST01278,Shalini Verma,Female,21.0,Mumbai,RTMU005,Lower Parel - Mulund,Lower Parel,Mulund,MH23CF4441,AC Sleeper,DRV0080,Aarav Bhat,2024-09-21,20:25,20:25,20:58,20:58,10.9,153,0.0,UPI,Corporate Portal,D8,Completed,NaN,5.0,84.6,Haze,0,Corporate Plan,iOS,true,No
1,TRP002439,BK14257322,CUST01290,Kalpana Chopra,Male,24.0,Hyderabad,RTHY038,HITEC City - Kondapur,HITEC City,Kondapur,TS27CF8651,AC Seater,DRV0068,Sanjay Joshi,2024-08-08,14:50,15:12,15:29,15:51,14.1,127,0.0,Debit Card,Kiosk,D7,Delayed-Completed,NaN,NaN,28.3,Heavy Rain,False,Weekly Pass,Android,True,0
2,TRP000793,BK67236768,CUST01285,Sai Naidu,Female,18.0,Pune,RTPU012,Aundh - Magarpatta,Aundh,Magarpatta,MH24CF3646,AC Seater,DRV0001,Lakshmi Patel,2024-03-19,08:15,08:15,09:42,09:42,27.6,234,19.0,Cash,Corporate Portal,D11,Completed,NaN,3.0,15.4,Cloudy,1,Single Ride,Android,Yes,No
3,TRP002802,BK59659578,CUST00625,Priya Patel,Female,23.0,Mumbai,RTMU010,Ghatkopar - Lower Parel,Ghatkopar,Lower Parel,MH13CF5853,AC Sleeper,DRV0036,Siddharth Kumar,2024-12-29,17:45,17:45,18:57,NaN,28.6,310,0.0,Debit Card,Corporate Portal,D11,No-show,Customer Request,NaN,68.1,Clear,No,Corporate Plan,Web,1,Yes
4,TRP001653,BK19231279,CUST00276,Shalini Kulkarni,F,20.0,Bangalore,RTBA024,Hebbal - Electronic City,Hebbal,Electronic City,KA22CF6238,AC Seater,DRV0045,Sunita Gupta,2024-05-15,13:00,13:40,13:35,14:15,17.8,202,0.0,UPI,Corporate Portal,A1,Delayed-Completed,NaN,NaN,23.7,Haze,No,Corporate Plan,iOS,True,0


## Phase 3 — Analyze (Steps 18–21)

Explore relationships and statistically test them.

### Step 18 — Univariate Analysis
**Numerical:** histogram, box plot, summary stats (mean, median, mode, std, skew, kurtosis).
**Categorical:** `value_counts()`, bar chart, pie chart.

In [177]:
num_cols_final = [
    "age",
    "distance_km",
    "fare_inr",
    "delay_minutes",
    "occupancy_pct",
    "net_revenue_inr"
]

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[f"Distribution of {c}" for c in num_cols_final]
)

for i, c in enumerate(num_cols_final):
    row = i // 3 + 1
    col = i % 3 + 1

    data = df[c].dropna()

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=data,
            histnorm="probability density",
            marker_color="#3b6fa0",
            opacity=0.75,
            showlegend=False,
            nbinsx=30
        ),
        row=row,
        col=col
    )

    # KDE Curve
    kde = gaussian_kde(data)
    x_range = np.linspace(data.min(), data.max(), 300)
    y_kde = kde(x_range)

    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=y_kde,
            mode="lines",
            line=dict(color="red", width=2),
            showlegend=False
        ),
        row=row,
        col=col
    )

fig.update_layout(
    template="plotly_white",
    height=800,
    width=1200,
    title="Distribution of Numerical Features",
    title_x=0.5,
    bargap=0.05
)

fig.show(config={"responsive": True})

In [178]:

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[f"Boxplot of {c}" for c in num_cols_final]
)

for i, c in enumerate(num_cols_final):
    row = i // 3 + 1
    col = i % 3 + 1

    fig.add_trace(
        go.Box(
            x=df[c].dropna(),
            name=c,
            marker_color="#7fa8d9",
            boxmean=False,
            showlegend=False
        ),
        row=row,
        col=col
    )

fig.update_layout(
    template="plotly_white",
    height=700,
    width=1200,
    title="Boxplots of Numerical Features",
    title_x=0.5
)

fig.show(config={"responsive": True})

In [179]:
summary_stats = df[num_cols_final].agg(["mean", "median", "std",
                                          lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
                                          "skew", "kurt"])
summary_stats.index = ["mean", "median", "std", "mode", "skew", "kurtosis"]
summary_stats

,age,distance_km,fare_inr,delay_minutes,occupancy_pct,net_revenue_inr
mean,29.445000,24.738375,317.678281,5.209056,57.855147,304.076229
median,29.000000,25.700000,311.000000,0.000000,57.900000,300.000000
std,7.501554,10.291220,156.005937,11.919960,24.508039,152.124734
mode,18.000000,8.400000,766.500000,0.000000,57.800000,766.500000
skew,0.359248,-0.132789,0.455582,2.517691,-0.008881,0.518000
kurtosis,-0.382112,-1.129204,-0.225114,5.669984,-1.209838,-0.093902


In [180]:

fig = make_subplots(
    rows=1,
    cols=3,
    specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "domain"}]],
    subplot_titles=[
        "Trips by City",
        "Trips by Bus Type",
        "Trip Status Split"
    ]
)

# Trips by City
city_counts = df["city"].value_counts()

fig.add_trace(
    go.Bar(
        x=city_counts.index,
        y=city_counts.values,
        marker_color="#3b6fa0",
        showlegend=False
    ),
    row=1,
    col=1
)

# Trips by Bus Type
bus_counts = df["bus_type"].value_counts()

fig.add_trace(
    go.Bar(
        x=bus_counts.index,
        y=bus_counts.values,
        marker_color="#5f8fc7",
        showlegend=False
    ),
    row=1,
    col=2
)

# Trip Status Split
status_counts = df["trip_status"].value_counts()

fig.add_trace(
    go.Pie(
        labels=status_counts.index,
        values=status_counts.values,
        textinfo="percent+label",
        showlegend=False
    ),
    row=1,
    col=3
)

fig.update_xaxes(tickangle=30, row=1, col=1)
fig.update_xaxes(tickangle=20, row=1, col=2)

fig.update_layout(
    template="plotly_white",
    height=450,
    width=1200,
    title="Categorical Feature Distributions",
    title_x=0.5
)

fig.show(config={"responsive": True})

### Step 19 — Bivariate Analysis
**Numerical vs Numerical:** scatter plot + correlation coefficient.
**Numerical vs Categorical:** box plot / bar plot of mean by category.
**Categorical vs Categorical:** cross-tabulation, stacked bar chart.

In [181]:
import plotly.express as px
from scipy import stats

fig = px.scatter(
    df,
    x="distance_km",
    y="fare_inr",
    opacity=0.3,
    color_discrete_sequence=["#3b6fa0"],
    title="Fare vs Distance"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Distance (km)",
    yaxis_title="Fare (INR)"
)

fig.show(config={"responsive": True})

# Pearson Correlation
pearson_r, pearson_p = stats.pearsonr(df["distance_km"], df["fare_inr"])
print(f"Pearson r = {pearson_r:.3f}  (p = {pearson_p:.2e})")

Pearson r = nan  (p = nan)


In [182]:


fig = px.box(
    df,
    x="bus_type",
    y="fare_inr",
    color="bus_type",
    color_discrete_sequence=px.colors.sequential.Blues,
    title="Fare by Bus Type"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Bus Type",
    yaxis_title="Fare (INR)",
    xaxis_tickangle=15,
    showlegend=False
)

fig.show(config={"responsive": True})

In [183]:
avg_rating = (
    df.groupby("city")["rating"]
      .mean()
      .sort_values()
      .reset_index()
)

fig = px.bar(
    avg_rating,
    x="rating",
    y="city",
    orientation="h",
    color_discrete_sequence=["#3b6fa0"],
    title="Average Rating by City"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Average Rating",
    yaxis_title="City",
    showlegend=False
)

fig.show(config={"responsive": True})

In [184]:
import plotly.graph_objects as go

ct = pd.crosstab(
    df["city"],
    df["trip_status"],
    normalize="index"
) * 100

fig = go.Figure()

for status in ct.columns:
    fig.add_trace(
        go.Bar(
            x=ct.index,
            y=ct[status],
            name=status
        )
    )

fig.update_layout(
    barmode="stack",
    template="plotly_white",
    title="Trip Status Composition by City (%)",
    title_x=0.5,
    xaxis_title="City",
    yaxis_title="% of Trips",
    xaxis_tickangle=30,
    legend_title="Trip Status",
    height=500,
    width=900
)

fig.show(config={"responsive": True})

# Display rounded percentage table
ct.round(1)

trip_status,Cancelled,Completed,Delayed-Completed,No-show
city,,,,
Bangalore,12.0,65.6,10.8,11.6
Chennai,12.6,65.6,11.8,10.1
Delhi NCR,13.8,64.1,13.2,8.9
Hyderabad,10.5,65.2,14.2,10.1
Mumbai,10.3,68.4,11.4,9.9
Pune,10.9,66.8,10.5,11.8


### Step 20 — Multivariate Analysis
Pair plot across numerical columns, correlation heatmap, grouped box plots with a categorical `hue`,
and a facet grid split by city.

In [185]:
import plotly.express as px

# Sample for a readable scatter matrix
sample_df = df.sample(min(600, len(df)), random_state=42)

fig = px.scatter_matrix(
    sample_df[
        ["distance_km", "fare_inr", "occupancy_pct", "delay_minutes"]
    ].dropna(),
    dimensions=[
        "distance_km",
        "fare_inr",
        "occupancy_pct",
        "delay_minutes"
    ],
    opacity=0.4,
    color_discrete_sequence=["#3b6fa0"],
    title="Pairwise Relationships Between Numerical Features"
)

fig.update_traces(
    diagonal_visible=True,
    marker=dict(size=5)
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=800,
    width=800
)

fig.show(config={"responsive": True})

In [186]:
# Correlation matrix
corr = df[num_cols_final].corr(numeric_only=True)

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="Blues",
    aspect="auto",
    title="Correlation Heatmap — Numerical Columns"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=600,
    width=700,
    coloraxis_colorbar_title="Correlation"
)

fig.update_xaxes(side="bottom")

fig.show(config={"responsive": True})

In [187]:
fig = px.box(
    df,
    x="city",
    y="fare_inr",
    color="bus_type",
    color_discrete_sequence=px.colors.sequential.Blues,
    title="Fare by City, grouped by Bus Type"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="City",
    yaxis_title="Fare (INR)",
    xaxis_tickangle=30,
    legend_title="Bus Type",
    height=500,
    width=1000,
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top"
    )
)

fig.show(config={"responsive": True})

In [188]:
import plotly.express as px

fig = px.histogram(
    df,
    x="delay_minutes",
    facet_col="city",
    facet_col_wrap=3,
    nbins=20,
    color_discrete_sequence=["#3b6fa0"],
    title="Delay Distribution by City"
)

# Remove "city=" from facet titles
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    height=750,
    margin=dict(t=90, b=70, l=60, r=30),
    bargap=0.05,
    showlegend=False
)

# Remove repeated axis titles
fig.for_each_xaxis(lambda axis: axis.update(title_text=""))
fig.for_each_yaxis(lambda axis: axis.update(title_text=""))

# Keep tick labels
fig.update_xaxes(matches=None)
fig.update_yaxes(matches="y")

# Add one global axis title
fig.add_annotation(
    text="Delay Minutes",
    x=0.5,
    y=-0.08,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=14)
)

fig.add_annotation(
    text="Count",
    x=-0.07,
    y=0.5,
    xref="paper",
    yref="paper",
    textangle=-90,
    showarrow=False,
    font=dict(size=14)
)

fig.show(config={"responsive": True})

### Step 21 — Hypothesis Testing
State H₀ (no effect/difference) and H₁ (there is one), then choose the test based on variable types,
checking normality (Shapiro-Wilk, on a sample since N is large) and equal variance (Levene's test)
first.

**21a — Numerical vs Numerical:** Pearson correlation test between `distance_km` and `fare_inr`.

- H₀: There is no linear correlation between distance and fare.
- H₁: There is a linear correlation between distance and fare.

In [189]:
r, p = stats.pearsonr(df["distance_km"], df["fare_inr"])
print(f"Pearson r = {r:.3f}, p-value = {p:.2e}")
print("Reject H0 -> significant correlation" if p < 0.05 else "Fail to reject H0")

Pearson r = nan, p-value = nan
Fail to reject H0


**21b — Numerical vs Categorical (2 groups):** t-test comparing `delay_minutes` between peak-hour
and non-peak-hour trips.

- H₀: Mean delay is the same for peak-hour and non-peak-hour trips.
- H₁: Mean delay differs between the two groups.

In [190]:
peak = df.loc[df["is_peak_hour"] == True, "delay_minutes"].dropna()
non_peak = df.loc[df["is_peak_hour"] == False, "delay_minutes"].dropna()

# Assumption checks (Shapiro on a sample, Levene for equal variance)
shapiro_peak = stats.shapiro(peak.sample(min(500, len(peak)), random_state=1))
levene_stat, levene_p = stats.levene(peak, non_peak)
print(f"Shapiro-Wilk (peak sample) p = {shapiro_peak.pvalue:.4f}")
print(f"Levene's test p = {levene_p:.4f}")

t_stat, t_p = stats.ttest_ind(peak, non_peak, equal_var=(levene_p > 0.05))
print(f"\nt-statistic = {t_stat:.3f}, p-value = {t_p:.4f}")
print(f"Peak mean delay = {peak.mean():.1f} min | Non-peak mean delay = {non_peak.mean():.1f} min")
print("Reject H0 -> delay differs by peak hour" if t_p < 0.05 else "Fail to reject H0")

Shapiro-Wilk (peak sample) p = 0.0000
Levene's test p = 0.6068

t-statistic = 0.550, p-value = 0.5824
Peak mean delay = 5.4 min | Non-peak mean delay = 5.1 min
Fail to reject H0


**21c — Numerical vs Categorical (3+ groups):** one-way ANOVA comparing `fare_inr` across
`bus_type`.

- H₀: Mean fare is the same across all bus types.
- H₁: At least one bus type has a different mean fare.

In [191]:
groups = [g["fare_inr"].dropna().values for _, g in df.groupby("bus_type")]
levene_stat, levene_p = stats.levene(*groups)
f_stat, anova_p = stats.f_oneway(*groups)

print(f"Levene's test p = {levene_p:.4f}")
print(f"ANOVA F = {f_stat:.2f}, p-value = {anova_p:.2e}")
print("Reject H0 -> fare differs by bus type" if anova_p < 0.05 else "Fail to reject H0")

df.groupby("bus_type")["fare_inr"].mean().round(1)

Levene's test p = 0.0000
ANOVA F = 86.53, p-value = 1.46e-53
Reject H0 -> fare differs by bus type


bus_type
AC Seater        276.0
AC Sleeper       343.8
Non-AC Seater    272.9
Premium AC       379.3
Name: fare_inr, dtype: float64

**21d — Categorical vs Categorical:** Chi-square test of independence between `city` and
`trip_status` (is cancellation/no-show behavior associated with city?).

- H₀: `city` and `trip_status` are independent.
- H₁: `city` and `trip_status` are associated.

In [170]:
contingency = pd.crosstab(df["city"], df["trip_status"])
chi2, chi_p, dof, expected = stats.chi2_contingency(contingency)

print(f"Chi-square = {chi2:.2f}, dof = {dof}, p-value = {chi_p:.4f}")
print("Reject H0 -> city and trip_status are associated" if chi_p < 0.05 else "Fail to reject H0")
contingency

Chi-square = 12.80, dof = 15, p-value = 0.6181
Fail to reject H0


trip_status,Cancelled,Completed,Delayed-Completed,No-show
city,,,,
Bangalore,64,351,58,62
Chennai,65,339,61,52
Delhi NCR,74,344,71,48
Hyderabad,53,330,72,51
Mumbai,57,378,63,55
Pune,60,369,58,65
